In [0]:
storage_account = "stgecommercedev"
storage_key = "**REDACTED**"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/olist"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/olist"
raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/olist"

print(silver_path)

abfss://silver@stgecommercedev.dfs.core.windows.net/olist


In [0]:
from pyspark.sql.functions import trim,col,upper,to_timestamp
from pyspark.sql.types import DoubleType,IntegerType

### clean ORDERS table

In [0]:
orders=spark.read.format("delta").load(f"{bronze_path}/orders_delta")
orders.limit(5).display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [0]:
orders_clean= orders\
    .dropDuplicates(["order_id"])\
    .dropna(subset=["order_id","customer_id","order_status"])\
    .withColumn("order_purchase_timestamp",to_timestamp(col("order_purchase_timestamp")))\
    .withColumn("order_delivered_customer_date",to_timestamp(col("order_delivered_customer_date")))\
    .withColumn("order_estimated_delivery_date",to_timestamp(col("order_estimated_delivery_date")))

print(f"Orders before cleaning: {orders.count()} and after cleaning: {orders_clean.count()}")

orders_clean.limit(5).display()

Orders before cleaning: 99441 and after cleaning: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,2018-03-17T15:32:31Z,2018-03-17 15:48:40,2018-03-20 21:08:28,null,2018-04-13T00:00:00Z
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,2018-03-09 19:08:26,2018-03-13 21:24:28,2018-04-11T12:53:50Z,2018-04-04T00:00:00Z
cc66dee6fbc18bb79903c3a2cc14ff52,19d3b3a2d4756af17603e2c35c7c2815,delivered,2018-04-12T14:37:29Z,2018-04-12 15:15:27,2018-04-16 16:23:53,2018-04-20T17:28:56Z,2018-05-07T00:00:00Z
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,2018-07-13 22:35:20,2018-07-24 19:07:00,2018-07-25T14:03:41Z,2018-07-31T00:00:00Z
edcc6b79e8394346ba3ba21b00b4055e,08aea10c40f606e52597486db2b56a81,delivered,2018-04-29T16:03:47Z,2018-04-29 16:15:25,2018-05-02 08:25:00,2018-05-11T23:12:12Z,2018-05-25T00:00:00Z


In [0]:
orders_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/orders/")

### clean ORDER_ITEMS table

In [0]:
items=spark.read.format("delta").load(f"{bronze_path}/order_items_delta/")
items.limit(5).display()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [0]:
items_clean=items\
    .dropDuplicates(["order_id","order_item_id"])\
    .dropna(subset=["order_id","product_id","price"])\
    .withColumn("price",col("price").cast(DoubleType()))\
    .withColumn("freight_value",col("freight_value").cast(DoubleType()))

print(f"Items before cleaning: {items.count()} and after cleaning: {items_clean.count()}")

items_clean.limit(5).display()

Items before cleaning: 112650 and after cleaning: 112650


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,2017-05-29 03:15:24,16.5,14.1
0016dfedd97fc2950e388d2971d718c7,1,4089861a1bd4685da70bddd6b4f974f1,a35124e2d763d7ca3fbe3b97d143200f,2017-05-05 10:05:12,49.75,20.8
0071ee2429bc1efdc43aa3e073a5290e,1,00ffe57f0110d73fd84d162252b2c784,53e4c6e0f4312d4d2107a8c9cddf45cd,2018-01-26 14:17:41,179.98,12.46
0115abf6b892040abfdd5bdfcb6b2c51,1,dc2410804cf782c5d87dbcd201b74e9b,897060da8b9a21f655304d50fd935913,2017-04-27 05:10:46,196.0,11.98
01e6f40a293da5a9dc11e49695c7541d,1,f7ac498363699addc5ff78eadee46977,2138ccb85b11a4ec1e37afbd1c8eda1f,2017-06-05 04:30:25,48.99,15.1


In [0]:
items_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/order_items/")

### clean ORDER_REVIEWS table

In [0]:
reviews=spark.read\
    .option("header",True)\
    .option("multiLine",True)\
    .option("quote","\"")\
    .option("escape","\"")\
    .csv(f"{raw_path}/olist_order_reviews_dataset.csv")

reviews_clean=reviews\
    .dropDuplicates(["review_id"])\
    .dropna(subset=["review_id","order_id"])
print(f"Reviews before cleaning: {reviews.count()} and after cleaning: {reviews_clean.count()}")


Reviews before cleaning: 99224 and after cleaning: 98410


In [0]:
reviews_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/order_reviews/")

### clean CUSTOMERS table

In [0]:
customers=spark.read.format("delta")\
    .load(f"{bronze_path}/customers_delta/")

customers_clean=customers\
    .dropDuplicates(["customer_id"])\
    .dropna(subset=["customer_id","customer_unique_id","customer_zip_code_prefix","customer_city","customer_state"])\
    .withColumn("customer_state",trim(col("customer_state")))

print(f"Customers before cleaning: {customers.count()} and after cleaning: {customers_clean.count()}")

Customers before cleaning: 99441 and after cleaning: 99441


In [0]:
customers_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/customers/")

### clean PRODUCTS table

In [0]:
products=spark.read.format("delta")\
    .load(f"{bronze_path}/products_delta/")

products_clean=products\
    .dropDuplicates(["product_id"])\
    .dropna(subset=["product_id"])
print(f"Products before cleaning: {products.count()} and after cleaning: {products_clean.count()}")
products_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/products/")

Products before cleaning: 32951 and after cleaning: 32951


### clean SELLERS table

In [0]:
sellers=spark.read.format("delta")\
    .load(f"{bronze_path}/sellers_delta/")

sellers_clean=sellers\
    .dropDuplicates(["seller_id"])\
    .withColumn("seller_zip_code_prefix",col("seller_zip_code_prefix").cast(IntegerType()))

In [0]:
sellers_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/sellers/")

### clean GEOLOCATION table

In [0]:
geo=spark.read.format("delta")\
    .load(f"{bronze_path}/geolocation_delta/")

geo_clean=geo\
    .withColumn("geolocation_lat",col("geolocation_lat").cast(DoubleType())).alias("Latitude")\
    .withColumn("geolocation_lng",col("geolocation_lng").cast(DoubleType())).alias("Longitude")

print(f"Geolocation before cleaning: {geo.count()} and after cleaning: {geo_clean.count()}")
geo_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/geolocation/")

Geolocation before cleaning: 1000163 and after cleaning: 1000163


### clean ORDER_PAYMENTS table

In [0]:
payments=spark.read.format("delta")\
    .load(f"{bronze_path}/order_payments_delta/")

payments_clean=payments\
    .dropna(subset=["order_id"])\
    .withColumn("payment_value",col("payment_value").cast(DoubleType()))

print(f"Payments before cleaning: {payments.count()} and after cleaning: {payments_clean.count()}")
payments_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/order_payments/")

Payments before cleaning: 103886 and after cleaning: 103886


### clean PRODUCT_CATEGORY table

In [0]:
prod_cat=spark.read.format("delta")\
    .load(f"{bronze_path}/product_category_delta/")

prod_cat_clean=prod_cat\
    .dropDuplicates(["product_category_name"])
print(f"Product categories before cleaning: {prod_cat.count()} and after cleaning: {prod_cat_clean.count()}")
prod_cat_clean.write.format("delta")\
    .mode("overwrite")\
    .save(f"{silver_path}/product_category/")

Product categories before cleaning: 71 and after cleaning: 71
